In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QESEM: функція Qiskit від Qedma
> **Note:** Функції Qiskit є експериментальною можливістю, доступною лише для користувачів IBM Quantum&reg; Premium Plan, Flex Plan та On-Prem (через IBM Quantum Platform API) Plan. Вони перебувають у стані попереднього випуску та можуть змінюватись.

## Огляд
Хоча квантові процесори значно вдосконалилися за останні роки, помилки через шуми та недосконалості наявного обладнання залишаються центральною проблемою для розробників квантових алгоритмів. У міру того як галузь наближається до квантових обчислень у масштабі корисності, які не піддаються класичній верифікації, рішення для усунення шумів із гарантованою точністю стають дедалі важливішими. Щоб подолати цю проблему, Qedma розробила QESEM (Quantum Error Suppression and Error Mitigation — придушення та пом'якшення квантових помилок), безперешкодно інтегровану в IBM Quantum Platform як [функцію Qiskit](/guides/functions).

За допомогою QESEM ти можеш запускати квантові схеми на зашумлених QPU та отримувати точні результати без похибок із мінімальними витратами часу QPU, близькими до фундаментальних меж. Для цього QESEM використовує набір патентованих методів, розроблених Qedma, для характеризації та зменшення помилок. Техніки зменшення помилок включають оптимізацію вентилів, транспіляцію з урахуванням шумів, придушення помилок (ES) та незміщене пом'якшення помилок (EM). Завдяки поєднанню цих методів на основі характеризації користувачі можуть отримувати надійні результати без похибок для загальних квантових схем великого об'єму, відкриваючи застосування, які неможливо реалізувати інакше.

Повний опис базових компонентів, а також демонстрацію у масштабі корисності наведено у статті [Reliable high-accuracy error mitigation for utility-scale quantum circuits.](https://arxiv.org/abs/2508.10997)
## Опис
Ти можеш використовувати функцію QESEM від Qedma для легкого оцінювання та виконання своїх схем із придушенням і пом'якшенням помилок, досягаючи більших об'ємів схем і вищої точності. Щоб використовувати QESEM, ти надаєш квантову схему, набір спостережуваних величин для вимірювання, цільову статистичну точність для кожної з них та вибраний QPU. Перед запуском схеми на цільову точність ти можеш оцінити необхідний час QPU на основі аналітичного розрахунку, який не вимагає виконання схеми. Коли ти задоволений оцінкою часу QPU, можна виконати схему за допомогою QESEM.

Під час виконання схеми QESEM запускає протокол характеризації пристрою, адаптований до твоєї схеми, формуючи надійну модель шуму для помилок, що виникають у схемі. На основі характеризації QESEM спочатку реалізує транспіляцію з урахуванням шумів для відображення вхідної схеми на набір фізичних кубітів і вентилів, що мінімізує шум, який впливає на цільову спостережувану величину. Сюди входять вентили, доступні нативно (CX/CZ на пристроях IBM&reg;), а також додаткові вентили, оптимізовані QESEM, що утворюють розширений набір вентилів QESEM. Потім QESEM запускає набір схем ES і EM на основі характеризації на QPU та збирає результати вимірювань. Вони класично постобробляються для отримання незміщеного математичного сподівання та смуги помилок для кожної спостережуваної величини, що відповідає запитаній точності.

![Огляд Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
Доведено, що QESEM забезпечує високоточні результати для різноманітних квантових застосувань та для найбільших об'ємів схем, досяжних сьогодні. QESEM пропонує такі функції для користувачів, продемонстровані в розділі бенчмарків нижче:
-	**Гарантована точність:** QESEM видає незміщені оцінки математичних сподівань спостережуваних величин. Його метод EM оснащений теоретичними гарантіями, які — разом із передовою характеризацією Qedma — забезпечують збіжність пом'якшення до виходу ідеальної схеми з точністю, заданою користувачем. На відміну від багатьох евристичних методів EM, схильних до систематичних помилок або зміщень, гарантована точність QESEM є ключовою для отримання надійних результатів у загальних квантових схемах та спостережуваних величинах.
-	**Масштабованість до великих QPU:** Час QPU для QESEM залежить від об'ємів схем, але в іншому не залежить від кількості кубітів. Qedma продемонструвала QESEM на найбільших квантових пристроях, доступних сьогодні, включаючи IBM Quantum Eagle на 127 кубітів та Heron на 133 кубіти.
-	**Незалежність від застосування:** QESEM продемонстровано для різноманітних застосувань, включаючи симуляцію гамільтоніана, VQE, QAOA та оцінку амплітуди. Користувачі можуть вводити будь-яку квантову схему та спостережувану величину для вимірювання та отримувати точні результати без похибок. Єдині обмеження диктуються специфікаціями обладнання та виділеним часом QPU, які визначають доступні об'єми схем та точність вихідних даних. На відміну від цього, багато рішень для зменшення помилок є специфічними для застосувань або включають неконтрольовані евристики, що робить їх непридатними для загальних квантових схем та застосувань.
-  **Розширений набір вентилів:** QESEM підтримує вентилі з дробовими кутами та надає оптимізовані Qedma вентилі $Rzz(\theta)$ з дробовими кутами на пристроях IBM Quantum Eagle. Цей розширений набір вентилів уможливлює ефективнішу компіляцію та відкриває об'єми схем, більші в до 2 разів порівняно зі стандартною компіляцією CX/CZ.
-	**Мультибазисні спостережувані:** QESEM підтримує вхідні спостережувані, складені з багатьох некомутуючих рядків Паулі, як-от загальні гамільтоніани. Вибір базисів вимірювання та оптимізація розподілу ресурсів QPU (вибірки та схеми) тоді виконуються автоматично QESEM для мінімізації необхідного часу QPU для запитаної точності. Ця оптимізація, що враховує надійність обладнання та частоти виконання, дає змогу запускати глибші схеми та отримувати вищу точність.
## Бенчмарки
QESEM було протестовано на широкому спектрі випадків використання та застосувань. Наступні приклади допоможуть тобі оцінити, які типи робочих навантажень можна запускати за допомогою QESEM.

Ключовою фігурою якості для кількісної оцінки складності як пом'якшення помилок, так і класичної симуляції для заданої схеми та спостережуваної величини є **активний об'єм**: кількість вентилів CNOT, що впливають на спостережувану величину в схемі. Активний об'єм залежить від глибини та ширини схеми, від ваги спостережуваної величини та від структури схеми, яка визначає світловий конус спостережуваної величини. Докладніше дивись у доповіді на [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM забезпечує особливо велику цінність у режимі великого об'єму, даючи надійні результати для загальних схем і спостережуваних величин.

![Активний об'єм](../docs/images/guides/qedma-qesem/active_volume.svg)

| Застосування    | Кількість кубітів | Пристрій | Опис схеми | Точність | Загальний час | Використання Runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Схема VQE  | 8              | Eagle (r3) | 21 загальний шар, 9 базисів вимірювання, 1D ланцюжок                    | 98%      | 35 хв      | 14 хв         |
| Kicked Ising   | 28              | Eagle (r3) | 3 унікальні шари × 3 кроки, 2D топологія heavy-hex                      | 97%     | 22 хв      | 4 хв          |
| Kicked Ising   | 28              | Eagle (r3) | 3 унікальні шари × 8 кроків, 2D топологія heavy-hex                     | 97%      | 116 хв      | 23 хв          |
| Троттеризована симуляція гамільтоніана   | 40  | Eagle (r3)            | 2 унікальні шари × 10 кроків Троттера, 1D ланцюжок                    | 97%      | 3 год     | 25 хв         |
| Троттеризована симуляція гамільтоніана   | 119  | Eagle (r3)           | 3 унікальні шари × 9 кроків Троттера, 2D топологія heavy-hex                    | 95%      | 6,5 год     | 45 хв         |
| Kicked Ising  | 136             | Heron (r2) | 3 унікальні шари × 15 кроків, 2D топологія heavy-hex                    | 99%      | 52 хв             | 9 хв           |

Точність тут вимірюється відносно ідеального значення спостережуваної величини: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, де '$\epsilon$' — абсолютна точність пом'якшення (задається вхідними даними користувача), а $\langle O \rangle_{ideal}$ — спостережувана на ідеальній схемі.
«Використання Runtime» вимірює використання бенчмарку в пакетному режимі (сума використання окремих завдань), тоді як «загальний час» вимірює використання в режимі сесії (астрономічний час експерименту), який включає додатковий класичний час та час зв'язку. QESEM доступний для виконання в обох режимах, щоб користувачі могли найкраще використовувати свої доступні ресурси.

Схеми Kicked Ising на 28 кубітів симулюють Дискретний часовий квазікристал, вивчений Shinjо та ін. (дивись [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) і [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) на трьох з'єднаних петлях ibm_kawasaki. Параметри схеми, взяті тут: $(\theta_x, \theta_z) = (0.9 \pi, 0)$, з феромагнітним початковим станом $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Вимірювана спостережувана величина — абсолютне значення намагніченості $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Експеримент Kicked Ising у масштабі корисності було проведено на 136 найкращих кубітах ibm_fez; цей конкретний бенчмарк виконувався при кліффордовому куті $(\theta_x, \theta_z) = (\pi, 0)$, при якому активний об'єм повільно зростає з глибиною схеми, що — разом із високою точністю пристрою — забезпечує високу точність при короткому часі виконання.

Схеми троттеризованої симуляції гамільтоніана призначені для моделі Ізінга в поперечному полі при дробових кутах: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ та $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ відповідно (дивись [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Схему у масштабі корисності було запущено на 119 найкращих кубітах ibm_brisbane, тоді як експеримент на 40 кубітів виконувався на найкращому доступному ланцюжку. Точність повідомляється для намагніченості; високоточні результати отримано також для спостережуваних величин більшої ваги.

Схема VQE була розроблена спільно з дослідниками Центру технологій та застосувань квантових обчислень Deutsches Elektronen-Synchrotron (DESY). Цільовою спостережуваною величиною тут був гамільтоніан, що складається з великої кількості некомутуючих рядків Паулі, що підкреслює оптимізовану продуктивність QESEM для мультибазисних спостережуваних. Пом'якшення було застосовано до класично оптимізованого ансатцу; хоча ці результати ще не опубліковані, результати такої ж якості будуть отримані для різних схем із подібними структурними властивостями.
## Початок роботи
Авторизуйся за допомогою свого [ключа API IBM Quantum Platform](http://quantum.cloud.ibm.com/) та вибери функцію Qiskit QESEM так. (Цей фрагмент коду передбачає, що ти вже [зберіг свій обліковий запис](/guides/functions#install-qiskit-functions-catalog-client) у локальному середовищі.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## Приклад
Щоб розпочати, спробуй цей базовий приклад оцінювання необхідного часу QPU для запуску QESEM для заданого `pub`:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Наступний приклад виконує завдання QESEM:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Ти можеш використовувати знайомі API Qiskit Serverless для перевірки статусу робочого навантаження функції Qiskit або отримання результатів:

In [ ]:
print(sample_job.status())
result = sample_job.result()

## Параметри функції
| Назва |  Тип | Опис | Обов'язковий | За замовчуванням |  Приклад |
| -----| ------| ------------| -------- | ------- | -------- |
| `pubs` | [EstimatorPubLike](/guides/primitive-input-output) |Це основний вхід. `Pub` містить 2–4 елементи: схему, одну або кілька спостережуваних, 0 або один набір значень параметрів та необов'язкову точність. Якщо точність не задано, буде використано `default_precision` з `options`|  Так|  Н/Д |  `[(circuit, [obs1,obs2,obs3], parameter_values, 0.03)]`  |
| `backend_name`| string|Назва бекенду для використання |Ні | QESEM вибере найменш завантажений пристрій, про який повідомляє IBM| `"ibm_fez"`|
| `instance` | string|  Ім'я хмарного ресурсу (CRN) інстансу для використання у відповідному форматі |  Ні |  Н/Д | `"CRN"`  |
| `options` | dictionary |Вхідні параметри. Дивись розділ **Параметри** для отримання докладнішої інформації. | Ні |  Дивись розділ **Параметри** для деталей.    |  `{ default_precision = 0.03, "max_execution_time" = 3600, "transpilation_level" = 0}`  |

### Параметри
| Параметр |  Значення | Опис | За замовчуванням |
| -----| -----------| -------- | ------- |
| `estimate_time_only` |  `"analytical"`  / `"empirical"` / None  | Коли встановлено, завдання QESEM лише розраховує оцінку часу QPU. Докладніший опис дивись нижче. Коли встановлено None, схема буде виконана з QESEM| None |
|`default_precision` | 0 < float | Застосовується до `pubs`, для яких не задано точність. Точність означає допустиму похибку математичних сподівань спостережуваних в абсолютному значенні. А саме, час роботи QPU для пом'якшення буде визначено таким чином, щоб забезпечити вихідні значення для всіх цікавих спостережуваних, що потрапляють у довірчий інтервал `1σ` цільової точності. Якщо надано кілька спостережуваних, пом'якшення виконуватиметься до досягнення цільової точності для кожної з них. | 0.02|
|`max_execution_time` | 0 < integer < 28 800 (8 годин)| Дає змогу обмежити час QPU, заданий у секундах, для всього процесу QESEM. Докладнішу інформацію дивись нижче.| 3 600 (одна година)|
| `transpilation_level` | 0 / 1 | Дивись опис нижче | 1|
| `execution_mode` | `"session"` / `"batch"` | Дивись опис нижче | "batch"|

> **Caution:** Оцінка часу QPU змінюється від одного бекенду до іншого. Тому під час виконання функції QESEM переконайся, що запускаєш її на тому самому бекенді, який було вибрано при отриманні оцінки часу QPU.

> **Note:** QESEM завершить роботу, коли досягне цільової точності або коли досягне `max_execution_time`, залежно від того, що настане раніше.

- `estimate_time_only` — цей прапорець дає змогу користувачам отримати оцінку часу QPU, необхідного для виконання схеми з QESEM.
    - Якщо встановлено `"analytical"`, верхня межа часу QPU розраховується без витрати ресурсів QPU. Ця оцінка має роздільну здатність 30 хвилин (наприклад, 30 хвилин, 60 хвилин, 90 хвилин тощо). Зазвичай вона є песимістичною і може бути отримана лише для одиночних спостережуваних Паулі або сум Паулі без перетинних носіїв (наприклад, Z0+Z1). Вона насамперед корисна для порівняння рівнів складності різних параметрів, наданих користувачем (схема, точність тощо).
    - Щоб отримати точнішу оцінку часу QPU, встанови цей прапорець на `"empirical"`. Хоча цей варіант вимагає запуску невеликої кількості схем, він забезпечує значно точнішу оцінку часу QPU. Ця оцінка має роздільну здатність 5 хвилин (наприклад, 20 хвилин, 25 хвилин, 30 хвилин тощо). Користувач може виконувати емпіричну оцінку часу в пакетному режимі або в режимі сесії. Докладніше дивись опис `execution_mode`. Наприклад, у пакетному режимі емпірична оцінка часу споживатиме менше 10 хвилин часу QPU.

- `max_execution_time`: дає змогу обмежити час QPU, заданий у секундах, для всього процесу QESEM. Оскільки остаточний час QPU, необхідний для досягнення цільової точності, визначається динамічно під час завдання QESEM, цей параметр дає змогу обмежити вартість експерименту. Якщо динамічно визначений час QPU менший за час, виділений користувачем, цей параметр не впливатиме на експеримент. Параметр `max_execution_time` особливо корисний у випадках, коли аналітична оцінка часу, надана QESEM перед початком завдання, є надто песимістичною і користувач хоче все одно ініціювати завдання пом'якшення. Після досягнення часового обмеження QESEM припиняє надсилати нові схеми. Схеми, що вже були надіслані, продовжують виконуватися (тому загальний час може перевищити обмеження на 30 хвилин), і користувач отримує оброблені результати зі схем, які виконалися до цього моменту. Якщо потрібно встановити обмеження часу QPU менше за аналітичну оцінку, зверніться до Qedma для отримання оцінки точності, досяжної в межах часового обмеження.

- `transpilation_level`: після надсилання схеми до QESEM вона автоматично готує кілька альтернативних транспіляцій схеми та вибирає ту, що мінімізує час QPU. Наприклад, альтернативні транспіляції можуть використовувати оптимізовані Qedma вентилі RZZ з дробовими кутами для зменшення глибини схеми. Звісно, всі транспіляції еквівалентні вхідній схемі з точки зору ідеального виходу. Щоб мати більший контроль над транспіляцією схеми, задай рівень транспіляції в `options`. Тоді як `"transpilation_level": 1` відповідає типовій поведінці, описаній вище, `"transpilation_level": 0` включає лише мінімальні необхідні зміни до оригінальної схеми; наприклад, «шарування» — організацію операцій схеми в «шари» одночасних двокубітних вентилів. Зауваж, що автоматичне відображення апаратного забезпечення на кубіти з високою точністю застосовується в будь-якому разі.

| transpilation_level | Опис |
|:-:|:--|
| `1` | Транспіляція QESEM за замовчуванням. Готує кілька альтернативних транспіляцій та вибирає ту, що мінімізує час QPU. Бар'єри можуть бути змінені на кроці шарування. |
| `0` | Мінімальна транспіляція: пом'якшена схема буде структурно близькою до вхідної. Схеми рівня 0 мають відповідати топології зв'язності пристрою та задаватися в термінах таких вентилів: CX, Rzz(α) та стандартних однокубітних вентилів (U, x, sx, rz тощо). Бар'єри будуть дотримані на кроці шарування. |

- `execution_mode` — користувач може вибрати запуск завдання QESEM у виділеній [сесії IBM](/guides/execution-modes#session-mode) або в кількох [пакетах IBM](/guides/execution-modes#batch-mode):
    -   **Режим сесії**: сесії дорожчі, але забезпечують менший час до отримання результату. Після початку сесії QPU зарезервовано виключно для завдання QESEM. Розрахунок використання включає як час, витрачений на виконання QPU, так і пов'язані класичні обчислення (виконувані QESEM та IBM). Функція Qiskit QESEM подбає про створення та закриття сесії автоматично. Для користувачів із необмеженим доступом до QPU (наприклад, локальні установки) рекомендується використовувати режим сесії для швидшого виконання QESEM.
    -   **Пакетний режим**: у пакетному режимі QPU звільняється під час класичних обчислень, що призводить до нижчого використання QPU. Оскільки пакетні завдання, як правило, тривають довше, існує більший ризик дрейфу обладнання; QESEM включає заходи для виявлення та компенсації дрейфів, зберігаючи надійність протягом тривалих запусків.

> **Note:** Операції бар'єру зазвичай використовуються для задання шарів двокубітних вентилів у квантових схемах. У рівні 0 QESEM зберігає шари, задані бар'єрами. У рівні 1 шари, задані бар'єрами, розглядаються як одна альтернатива транспіляції при мінімізації часу QPU.
### Вихідні дані
Виходом функції Circuit є [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), який містить два поля:

- Один об'єкт [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Його можна індексувати безпосередньо з `PrimitiveResult`.

- Метадані на рівні завдання.

Кожен `PubResult` містить поле `data` та поле `metadata`.

- Поле `data` містить щонайменше масив математичних сподівань (`PubResult.data.evs`) та масив стандартних похибок (`PubResult.data.stds`). Воно також може містити більше даних залежно від використаних параметрів.

- Поле `metadata` містить метадані на рівні PUB (`PubResult.metadata`).

Наступний фрагмент коду описує, як отримати оцінку часу QPU (коли `estimate_time_only` встановлено):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

Наступний фрагмент коду демонструє, як отримати результати пом'якшення (коли `estimate_time_only` не встановлено) та метрики виконання. Вони містять важливі дані, що дають змогу глибше зрозуміти, як різні параметри впливають на виконання QESEM. Це також може бути корисним при написанні статті на основі твоїх досліджень.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## Отримання повідомлень про помилки
Якщо статус твого робочого навантаження — ERROR, використай job.result() для отримання повідомлення про помилку таким чином:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem)
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199.](https://arxiv.org/pdf/2507.01199)
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997.](https://arxiv.org/pdf/2508.10997)


</Admonition>